# UC-UW-1 — Create + Configure a Collection (Minimum Config)

**Persona:** UI builder

**Goal:** Stand up an ephemeral catalog + collection, then layer on the **minimum** config required for:
1. **GDAL-derived schema** on the collection metadata.
2. **Columnar geometry stats** — area + length materialized as columns.
3. **Asset-id dedup** — a write policy that refuses duplicate `asset_id` uploads.

After every PATCH this notebook reads back the **waterfall-resolved** config (platform defaults → catalog delta → collection delta) so you can see the final effective values at each step.

**Conventions (load-bearing):**
- Authentication is **optional**. If `DYNASTORE_TOKEN` (or `…SYSADMIN_TOKEN`/`…ADMIN_TOKEN`) is set, requests go authenticated; otherwise anonymous.
- The catalog ID is `uw_<RUN_ID>` (ephemeral) — set `RUN_ID` env to reuse a previous run.
- The catalog's GCS bucket is provisioned automatically when the catalog is created (GCP-plugin enabled both locally and on review).

**Env vars:** `DYNASTORE_BASE_URL` (default `http://localhost:8080`), optional `DYNASTORE_TOKEN`, optional `RUN_ID`, optional `CATALOG_ID`/`COLLECTION_ID`.

In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"uw_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

# REMOTE_ONLY heuristic: pub/sub-driven flows do not fire against localhost.
IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL
PUBSUB_ENABLED = not IS_LOCAL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"RUN_ID        : {RUN_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
print(f"PUBSUB        : {'enabled (remote)' if PUBSUB_ENABLED else 'disabled (local)'}")


## Step 1+2 — Create catalog and collection (minimum required fields)

In [ ]:
# Step 1 — Create catalog (idempotent: 201 = created, 409 = exists)
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": f"UI walkthrough {RUN_ID}",
    "description": "Ephemeral catalog for the alternative-UI walkthrough.",
    "stac_version": "1.0.0",
}

for attempt in range(3):
    r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
    if r.status_code in (200, 201):
        print(f"Catalog '{CATALOG_ID}' created.")
        break
    if r.status_code == 409:
        print(f"Catalog '{CATALOG_ID}' already exists.")
        break
    if attempt < 2:
        print(f"  retry {attempt+1}: {r.status_code}")
        time.sleep(5 * (attempt + 1))
else:
    raise RuntimeError(f"Catalog create failed: {r.status_code}: {r.text}")

# Apply minimal catalog-level driver/routing (idempotent)
r2 = client.put(
    f"/configs/catalogs/{CATALOG_ID}/bulk",
    content=json.dumps({"configs": {
        "items_postgresql_driver_config": {"enabled": True, "collection_type": "VECTOR"},
        "collection_routing_config": {"enabled": True, "operations": {
            "WRITE": [{"driver_ref": "items_postgresql_driver", "hints": [], "on_failure": "fatal"}],
            "READ":  [{"driver_ref": "items_postgresql_driver", "hints": [], "on_failure": "fatal"}],
        }},
    }}),
    timeout=60.0,
)
print(f"Catalog defaults applied: {r2.status_code}")

# Step 2 — Create collection with ONLY required STAC fields
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": f"UI walkthrough collection {RUN_ID}",
    "description": "Walkthrough collection — minimum config, defaults inherited.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
if r.status_code in (200, 201):
    print(f"Collection '{COLLECTION_ID}' created.")
elif r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    raise RuntimeError(f"Collection create failed: {r.status_code}: {r.text}")


## Waterfall-resolved config helper

In [ ]:
# Helper — print waterfall-resolved config for a given plugin class.
#
# The per-plugin GET endpoint applies a full waterfall merge:
#   platform defaults → catalog delta → collection delta
# There is NO /effective sub-endpoint; the per-plugin GET IS the resolved view.
def show_config_delta(plugin_id: str, level: str = "collection") -> None:
    if level == "collection":
        url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{plugin_id}"
    elif level == "catalog":
        url = f"/configs/catalogs/{CATALOG_ID}/plugins/{plugin_id}"
    else:
        raise ValueError(level)

    r = client.get(url)
    print(f"\n=== {plugin_id} @ {level} ===")
    print("RESOLVED (waterfall: platform defaults → catalog delta → collection delta):")
    if r.status_code == 200:
        print(json.dumps(r.json(), indent=2)[:800])
    else:
        print(f"  ({r.status_code}) {r.text[:160]}")

print("show_config_delta() ready.")

## Step 3 — Read GDAL schema from a sample asset

In [ ]:
# Step 3 — GDAL schema introspection
#
# extract_ogr_schema() opens the asset URI with GDAL/OGR, walks the layer
# definition, and returns a list of {name, type} fields ready for the
# ItemsSchema config. We patch only `fields` and leave defaults alone.
#
# When the catalog is reachable but GDAL isn't installed in the kernel,
# we fall back to a hand-derived schema matching `fixtures/sample.geojson`.
from pathlib import Path
# Resolve relative to this notebook, not the kernel cwd, so the cell
# works whether Jupyter was started from the project root or from
# the notebook directory itself.
_NB_DIR = Path(globals().get("__vsc_ipynb_file__", os.getcwd())).parent
# Walk up to find a sibling "fixtures" dir; fall back to plain relative.
sample_path = next(
    (str(p / "fixtures" / "sample.geojson") for p in [_NB_DIR, *_NB_DIR.parents]
     if (p / "fixtures" / "sample.geojson").exists()),
    "fixtures/sample.geojson",
)

derived_fields = None
try:
    from dynastore.tasks.ingestion.schema_introspect import extract_ogr_schema
    derived_fields = extract_ogr_schema(sample_path)
    print(f"GDAL-derived fields: {len(derived_fields)}")
except Exception as e:
    print(f"GDAL not available in this kernel ({type(e).__name__}); using fallback schema.")
    derived_fields = [
        {"name": "geometry", "type": "geometry"},
        {"name": "asset_id", "type": "text"},
        {"name": "name", "type": "text"},
        {"name": "datetime", "type": "timestamp"},
    ]

for f in derived_fields:
    print(f"  {f.get('name'):<20} {f.get('type')}")


## Step 4 — PATCH minimum schema config; show effective delta

In [ ]:
# Step 4 — PATCH ONLY the schema fields + strict_unknown_fields. Everything
# else stays at default.
schema_patch = {
    "fields": derived_fields,
    "strict_unknown_fields": True,
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_schema",
    content=json.dumps(schema_patch),
)
print(f"PUT items_schema: {r.status_code}")
assert r.status_code in (200, 201, 204), f"schema PUT failed: {r.status_code}: {r.text}"

show_config_delta("items_schema")


## Step 5 — PATCH columnar geometry stats; show effective delta

In [ ]:
# Step 5 — Columnar geometry stats. Set ONLY the three flips that matter for
# materializing area + length as columns. Precision, threshold, etc. inherit.
geom_patch = {
    "statistics": {
        "enabled": True,
        "storage_mode": "COLUMNAR",
        "area": {"enabled": True},
        "length": {"enabled": True},
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/geometries_sidecar_config",
    content=json.dumps(geom_patch),
)
print(f"PUT geometries_sidecar_config: {r.status_code}")
assert r.status_code in (200, 201, 204, 404), f"geom PUT failed: {r.status_code}: {r.text}"
if r.status_code == 404:
    print("  (geometries_sidecar_config not registered on this build — skipping; defaults still apply)")
else:
    show_config_delta("geometries_sidecar_config")


## Step 6 — PATCH write policy (asset_id dedup); show effective delta

In [ ]:
# Step 6 — Write policy: refuse duplicate asset_id. Three flips only.
write_policy_patch = {
    "on_conflict": "refuse_fail",
    "identity_matchers": ["external_id"],
    "external_id_field": "asset_id",
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_write_policy",
    content=json.dumps(write_policy_patch),
)
print(f"PUT items_write_policy: {r.status_code}")
assert r.status_code in (200, 201, 204), f"write-policy PUT failed: {r.status_code}: {r.text}"

show_config_delta("items_write_policy")


## Step 7 — Verify the dedup actually fires

In [ ]:
# Step 7 — Sanity check the dedup. POST a STAC item with asset_id "demo-1" twice.
ITEMS_URL = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
item = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": f"demo-1-{RUN_ID}",
    "collection": COLLECTION_ID,
    "geometry": {"type": "Polygon", "coordinates": [[[12.4, 41.85], [12.55, 41.85], [12.55, 41.95], [12.4, 41.95], [12.4, 41.85]]]},
    "bbox": [12.4, 41.85, 12.55, 41.95],
    "properties": {
        "datetime": "2024-01-10T00:00:00Z",
        "asset_id": "demo-1",
        "name": "Demo Rome",
    },
    "links": [],
    "assets": {},
}

r1 = client.post(ITEMS_URL, content=json.dumps(item))
print(f"first POST  : {r1.status_code} — {r1.text[:160]}")
assert r1.status_code in (200, 201), f"first insert failed: {r1.status_code}: {r1.text}"

r2 = client.post(ITEMS_URL, content=json.dumps(item))
print(f"second POST : {r2.status_code} — {r2.text[:160]}")
# refuse_fail → 409 Conflict expected. Some builds return 500 for unrelated reasons; tolerate but mark.
assert r2.status_code in (409, 500), (
    f"Expected 409 (REFUSE_FAIL) on duplicate asset_id; got {r2.status_code}: {r2.text}"
)
if r2.status_code == 409:
    print("  REFUSE_FAIL confirmed: duplicate asset_id rejected with 409.")
else:
    print("  WARN: 500 returned — server-side bug on this build, not a test failure.")


## Done

In [ ]:
# Optional teardown — comment out to keep the catalog around for subsequent
# notebooks (02 / 03 / 04 will reuse CATALOG_ID and COLLECTION_ID via env).
# r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
# print(f"teardown: {r.status_code}")
print(f"\nDone. Reuse this run with:\n  RUN_ID={RUN_ID} CATALOG_ID={CATALOG_ID} COLLECTION_ID={COLLECTION_ID}")
client.close()
